[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/17_profiling_pyroscope/17_proyecto_profiling_pyroscope.ipynb)

# Proyecto 17 · Encuentra y arregla el cuello de botella de CPU de SupportOps con perfiles

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 2 h |
| **Servicios de Compose** | `profiling` (Pyroscope) para los pasos 6-7 (opcionales: sin él se usan datos pregrabados). Nada más |
| **RAM aproximada** | < 0,5 GB |
| **Prerrequisitos** | Lección `17_profiling_pyroscope`; módulos 03 y 16 |
| **Conexión** | Misma ruta caliente de la lección (`HotPathInstrumentor`). Aquí **tú** escribes el diagnóstico y el arreglo; el SLO de coste del módulo 16 es lo que justifica el trabajo |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El tráfico se ha multiplicado por 8 y la factura de cómputo del agente (no la del LLM: la de **CPU de tus pods**) se ha triplicado.
Los SLO de latencia siguen verdes porque el proveedor LLM domina el tiempo de petición, y por eso nadie lo había notado. Producto pide: *encuentra qué código quema la CPU, arréglalo sin cambiar el comportamiento,
demuéstralo con perfiles y deja una guarda para que no vuelva a pasar.*

## Datos

Tráfico sintético de SupportOps (semilla 42, `mock`): `run_workload("slow", n=…)` ejecuta tickets reales con CPU real; no hay datos externos. Con Pyroscope en `:4040` empujarás tus perfiles; sin él, la captura pregrabada cubre los mismos pasos.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `top_function` | función con más *self* del perfil lento | `scan_pii_slow` con > 50 % (la solución mide ≈ 68 %) |
| R2 | `stage_report` | reparto por etapa del perfil frente a los cronómetros | diferencia < 0,15 por etapa; `pii_scan` ∈ [0,50; 0,85] |
| R3 | `my_scan` | idéntico a `scan_pii_slow` en 5 textos y rapidez sobre el payload real (≈ 120 KB) | ≥ 5× (la solución: > 15×; una regex sola da ≈ 3-4×) |
| R4 | `my_tokens` | idéntico a `count_tokens_slow` en 5 textos y rapidez | ≥ 3× (la solución: ≈ 5×) |
| R5 | `run_fixed` | mismos estados y tokens, correos ± 1 %, CPU total | ≥ 3× menos (la solución: ≈ ×5) |
| R6 | `predict_speedup` | Amdahl con las fracciones medidas | lo medido ∈ [0,4×; 1,1×] del tope (la solución: ≈ 0,65×) |
| R7 | `push_profile`/`read_top` (con Pyroscope) | el perfil «antes» tiene `scan_pii_slow` arriba; el «después» ya no | exacto |
| R8 | `worst_ticket` | ticket más caro y su función principal vía etiquetas | `scan_pii_slow` |
| R9 | `cpu_guard` | falla la variante lenta y aprueba la corregida (presupuesto = ¼ de la línea base) | exacto |

> 📏 Las comparaciones usan **ratios** (no ms absolutos), así el resultado no depende de la velocidad de tu máquina.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import importlib.util, subprocess
if importlib.util.find_spec("pyroscope") is None:       # SDK oficial de Python (pyroscope-io) y su integración con OpenTelemetry
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyroscope-io", "pyroscope-otel"], check=False)

import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, gzip, inspect, json, math, time
from IPython.display import Markdown, display
from supportops import profiling as P
from supportops.profiling import PyroscopeClient, SamplingProfiler, run_workload
STACK = check_stack(["pyroscope"])                       # Pyroscope es lo único que NECESITA este módulo; Tempo/Collector solo para la sección de trazas
PYRO = PyroscopeClient()
PYRO_UP = PYRO.ready()
RUN = str(int(time.time()))                               # etiqueta única por ejecución: así el notebook no mezcla perfiles de ejecuciones anteriores
HAVE_SDK = P.have_pyroscope_sdk()
REC = P.load_recorded(str(ROOT / "17_profiling_pyroscope" / "recorded" / "capture_seed42.json.gz"))   # modo degradado: medido con este mismo código
N = 60 if FAST_DEV_RUN else 120                          # tickets por experimento (cada ejecución «lenta» cuesta ≈ 38 ms de CPU por ticket)
print(f"Pyroscope en vivo: {PYRO_UP} · SDK pyroscope-io: {HAVE_SDK} · tickets por experimento: {N}")

## Paso 0 · Línea base (te la damos hecha)

Ejecutamos la variante lenta bajo el perfilador a mano (`mode="cpu"`, 300 Hz) con etiquetas por ticket. `BASE` son los cronómetros por etapa; `ST_SLOW` las pilas; `PP` el perfilador con etiquetas.

In [ ]:
N_P = N                                                    # tickets por medición
from contextlib import contextmanager
class TicketTag(Instrumentor):
    def __init__(self, p): self.p = p
    @contextmanager
    def agent_run(self, ctx):
        self.p.set_tag("ticket", ctx.ticket.ticket_id)
        try: yield
        finally: self.p.clear_tag("ticket")
PP = SamplingProfiler(hz=300, mode="cpu")
BASE = run_workload("slow", n=N_P, profiler=PP, extra=[TicketTag(PP)])
ST_SLOW = PP.stacks()
SERVICE = f"supportops-p17-{int(time.time())}"
print(f"{BASE.n_tickets} tickets · {BASE.ms_per_ticket:.1f} ms de CPU/ticket · {PP.total} muestras · cronómetros por etapa (s):", {k: round(v, 2) for k, v in BASE.stage_s.items()})

## Paso 1 · ¿Dónde está el cuello de botella? (TODO)

Escribe `top_function(stacks)`: devuelve `(nombre_corto_de_la_función, fracción_de_self)` de la función con más tiempo **propio** (*self*) de un perfil `Counter[pila→muestras]`.

<details>
<summary>🪜 Pista 1</summary>

La función hoja de cada pila es la que gasta el tiempo propio: `P.self_time` ya cuenta muestras por hoja.

</details>

<details>
<summary>🪜 Pista 2</summary>

`fn, n = P.self_time(stacks).most_common(1)[0]`; divide `n` por la suma de todos los self.

</details>

In [ ]:
def top_function(stacks):
    # TODO: usa P.self_time(stacks) (Counter función→muestras self); nombre corto = parte tras el último "."
    raise NotImplementedError("TODO 1")

try:
    print(top_function(ST_SLOW))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 2 · ¿Y el reparto por etapa? (TODO)

Escribe `stage_report(stacks, stats)` → `DataFrame` con columnas `etapa`, `perfil` (fracción según el perfil) y `cronómetro` (verdad de referencia) para `pii_scan`, `tokens`, `serialize`, `embed_rerank`. Usa `P.stage_shares` y `stats.stage_share()`.

<details>
<summary>🪜 Pista 1</summary>

Recorre las cuatro etapas y construye una fila por etapa con `.get(etapa, 0.0)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`pd.DataFrame([{"etapa": s, "perfil": sp.get(s, 0.0), "cronómetro": st.get(s, 0.0)} for s in etapas])`.

</details>

In [ ]:
def stage_report(stacks, stats):
    # TODO: P.stage_shares(stacks) -> dict etapa->fracción (incluye 'other'); stats.stage_share() -> fracción entre las 4 etapas cronometradas
    raise NotImplementedError("TODO 2")

try:
    display(stage_report(ST_SLOW, BASE).round(3))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 3 · El arreglo (1/2): escáner de PII (TODO)

Escribe `my_scan(text)` → `{"emails": int, "cards": int}` con el **mismo resultado** que `P.scan_pii_slow` y **≥ 5× más rápido** sobre el payload real (≈ 120 KB, que se repite casi idéntico en cada llamada al LLM). Las tarjetas se validan con Luhn (`P.luhn_ok`). **Mide primero** qué te da una regex sola (spoiler: ≈ 3-4× en solitario) y piensa qué trabajo se está repitiendo.

<details>
<summary>🪜 Pista 1</summary>

Recorrer carácter a carácter en Python es lo lento; el motor de regex lo hace en C. Pero con una regex sola sigues re-escaneando 120 KB que casi no cambian: ¿puedes recordar lo ya escaneado?

</details>

<details>
<summary>🪜 Pista 2</summary>

Un correo o una tarjeta nunca cruzan un salto de línea: divide el texto en líneas (`text.split("\n")`), escanea cada línea con una función con `@functools.lru_cache` y suma. Correos: `[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}` · tarjetas: `\d(?:[ -]?\d){12,18}` + `P.luhn_ok(re.sub(r"\D", "", m.group()))`.

</details>

In [ ]:
def my_scan(text):
    # TODO: regex precompiladas a nivel de módulo; tarjetas: 13-19 dígitos con espacios/guiones opcionales + P.luhn_ok.
    #       Para llegar a 5×: ¿qué parte del texto no cambia entre llamadas?
    raise NotImplementedError("TODO 3")

try:
    print(my_scan("a@b.org y 4111 1111 1111 1111"), P.scan_pii_slow("a@b.org y 4111 1111 1111 1111"))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 4 · El arreglo (2/2): contador de tokens (TODO)

Escribe `my_tokens(text)` con el mismo resultado que `P.count_tokens_slow` (cada palabra cuenta `ceil(len/4)` trozos) y ≥ 3× más rápido: la misma idea que en el paso 3.

<details>
<summary>🪜 Pista 1</summary>

El techo de la división entera: `-(-a // b)`; y el truco de caché del paso anterior sirve aquí igual.

</details>

<details>
<summary>🪜 Pista 2</summary>

`@functools.lru_cache` sobre `lambda line: sum(-(-len(w) // 4) for w in line.split())` y suma por líneas.

</details>

In [ ]:
def my_tokens(text):
    # TODO: suma de ceil(len(palabra)/4) para cada palabra de text.split(); con caché por línea
    raise NotImplementedError("TODO 4")

try:
    print(my_tokens("hola mundo cruel"), P.count_tokens_slow("hola mundo cruel"))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 5 · Aplica el arreglo y mide (TODO)

Escribe `run_fixed(n)` que ejecute la variante `slow` conectando **tus** funciones con los ganchos `scan_fn`/`tokens_fn` de `run_workload`; y `predict_speedup(fraccion)` con el tope de Amdahl (`s → ∞`).

<details>
<summary>🪜 Pista 1</summary>

`run_workload` acepta `scan_fn=` y `tokens_fn=`: funciones `texto -> resultado` que sustituyen a las lentas.

</details>

<details>
<summary>🪜 Pista 2</summary>

`P.amdahl_speedup(fraccion, 1e9)` te da el tope; compáralo con `BASE.cpu_s / fx.cpu_s`.

</details>

In [ ]:
def run_fixed(n):
    # TODO: run_workload("slow", n=n, scan_fn=..., tokens_fn=...)
    raise NotImplementedError("TODO 5a")

def predict_speedup(fixed_share):
    # TODO: tope de Amdahl cuando la parte arreglada (fracción fixed_share) se vuelve gratis: P.amdahl_speedup(f, 1e9)
    raise NotImplementedError("TODO 5b")

try:
    fx = run_fixed(N_P)
    f = BASE.stage_share()["pii_scan"] + BASE.stage_share()["tokens"]
    print(f"CPU: {BASE.ms_per_ticket:.1f} → {fx.ms_per_ticket:.1f} ms/ticket (×{BASE.cpu_s / fx.cpu_s:.1f}) · tope de Amdahl con f={f:.2f}: ×{predict_speedup(f):.1f}")
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 6 · Perfiles en Pyroscope (TODO; necesita el perfil `profiling`)

Escribe `push_profile(prof, service, variant)` (empuja `prof.to_pprof()` con `PyroscopeClient.push_connect`) y `read_top(service, variant)` (espera a que exista y devuelve el nombre corto de la función con más *self* según **Pyroscope**). Luego empuja el perfil «antes» (`PP`) y el «después».

<details>
<summary>🪜 Pista 1</summary>

La API de la lección §6.1: `push_connect(labels, pprof_gz)`; el `__name__` ya lo pone por ti.

</details>

<details>
<summary>🪜 Pista 2</summary>

`sel = '{service_name="%s",variant="%s"}' % (service, variant)`; `c.wait_for(sel, timeout_s=90)`; `c.stacks(sel)`.

</details>

In [ ]:
def push_profile(prof, service, variant):
    # TODO: P.PyroscopeClient().push_connect({"service_name": service, "variant": variant}, prof.to_pprof())
    raise NotImplementedError("TODO 6a")

def read_top(service, variant):
    # TODO: selector '{service_name="...",variant="..."}' -> PyroscopeClient().wait_for(sel) -> .stacks(sel) -> P.self_time(...).most_common(1)
    raise NotImplementedError("TODO 6b")

try:
    if PYRO_UP:
        PF = SamplingProfiler(hz=300, mode="cpu"); run_workload("slow", n=N_P, profiler=PF, scan_fn=my_scan, tokens_fn=my_tokens)
        push_profile(PP, SERVICE, "slow"); push_profile(PF, SERVICE, "fixed")
        print("antes →", read_top(SERVICE, "slow"), "· después →", read_top(SERVICE, "fixed"))
    else:
        print("Pyroscope no está arriba: este paso se omite (docker compose --profile profiling up -d). R7 no puntuará.")
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 7 · El ticket más caro (TODO)

Escribe `worst_ticket(prof)` → `(ticket_id, función_principal_de_ese_ticket)` usando las etiquetas `ticket` que el paso 0 añadió a cada muestra (`prof.tag_values("ticket")` y `prof.stacks(ticket=...)`).

<details>
<summary>🪜 Pista 1</summary>

`prof.tag_values('ticket')` es un `Counter` ticket→muestras.

</details>

<details>
<summary>🪜 Pista 2</summary>

`prof.stacks(ticket=t)` filtra las pilas de ese ticket; después `P.self_time(...)`.

</details>

In [ ]:
def worst_ticket(prof):
    # TODO: ticket con más muestras y la función con más self de SUS pilas
    raise NotImplementedError("TODO 7")

try:
    print(worst_ticket(PP))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Paso 8 · Una guarda para que no vuelva (TODO)

Escribe `cpu_guard(ms_per_ticket, baseline_ms, max_ratio=0.25)`: `True` si el coste actual cabe en el presupuesto (`max_ratio` × línea base). Es la versión mínima de un test de regresión de rendimiento en CI.

<details>
<summary>🪜 Pista 1</summary>

Es una comparación; el presupuesto se expresa como ratio para no depender de la máquina.

</details>

In [ ]:
def cpu_guard(ms_per_ticket, baseline_ms, max_ratio=0.25):
    # TODO
    raise NotImplementedError("TODO 8")

try:
    print("lenta pasa la guarda:", cpu_guard(BASE.ms_per_ticket, BASE.ms_per_ticket), "· corregida pasa:", cpu_guard(run_fixed(N_P).ms_per_ticket, BASE.ms_per_ticket))
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## Rúbrica · ejecútala cuando quieras

In [ ]:
import re as _re
def run_rubric(live=PYRO_UP):
    R = {}
    def chk(name, fn):
        try:
            R[name] = bool(fn())
        except NotImplementedError:
            R[name] = None
        except Exception as e:
            R[name] = False; print(f"  {name}: {type(e).__name__}: {e}")
    PAYLOAD = P.serialize_slow(P.policy_text(), [])        # el texto que de verdad se escanea en cada llamada al LLM (≈ 120 KB)
    texts = ["Escribe a ana.lopez@example.com o a x@y.org", "tarjeta 4111 1111 1111 1111 y 4111111111111112", "nada que ver aquí", P.policy_text(), PAYLOAD]
    chk("R1 top_function identifica el cuello de botella (scan_pii_slow, > 50 % del self)", lambda: (lambda r: r[0] == "scan_pii_slow" and r[1] > 0.5)(top_function(ST_SLOW)))
    chk("R2 stage_report: perfil y cronómetros difieren < 0,15 en cada etapa y pii_scan ∈ [0,50; 0,85]",
        lambda: (lambda d: (abs(d["perfil"] - d["cronómetro"]) < 0.15).all() and 0.50 <= d.set_index("etapa").loc["pii_scan", "cronómetro"] <= 0.85)(stage_report(ST_SLOW, BASE)))
    chk("R3 my_scan: mismo resultado que scan_pii_slow en 5 textos y ≥ 5× más rápido sobre el payload real", lambda: all(my_scan(t) == P.scan_pii_slow(t) for t in texts) and
        (lambda a, b: a / b >= 5)(*(lambda f: (min(__import__("timeit").repeat(lambda: P.scan_pii_slow(PAYLOAD), number=3, repeat=3)), min(__import__("timeit").repeat(lambda: f(PAYLOAD), number=3, repeat=3))))(my_scan)))
    chk("R4 my_tokens: igual que count_tokens_slow en 5 textos y ≥ 3× más rápido", lambda: all(my_tokens(t) == P.count_tokens_slow(t) for t in texts) and
        (lambda a, b: a / b >= 3)(*(lambda f: (min(__import__("timeit").repeat(lambda: P.count_tokens_slow(PAYLOAD), number=3, repeat=3)), min(__import__("timeit").repeat(lambda: f(PAYLOAD), number=3, repeat=3))))(my_tokens)))
    def r5():
        fx = run_fixed(N_P)
        same = [r.status for r in fx.results] == [r.status for r in BASE.results] and [r.input_tokens for r in fx.results] == [r.input_tokens for r in BASE.results]
        close = abs(fx.pii_found["emails"] - BASE.pii_found["emails"]) <= 0.01 * BASE.pii_found["emails"]
        return same and close and BASE.cpu_s / fx.cpu_s >= 3
    chk("R5 run_fixed: mismos estados/tokens, correos ±1 % y CPU ≥ 3× menor", r5)
    def r6():
        fx = run_fixed(N_P)
        fr = BASE.cpu_fraction()
        pred = predict_speedup(fr["pii_scan"] + fr["tokens"])
        meas = BASE.cpu_s / fx.cpu_s
        return 0.4 * pred <= meas <= 1.1 * pred
    chk("R6 predict_speedup (Amdahl): lo medido queda entre 0,4× y 1,1× del tope", r6)
    if live:
        chk("R7 Pyroscope: el perfil antes→top scan_pii_slow y el de después ya no", lambda: (read_top(SERVICE, "slow") == "scan_pii_slow") and (read_top(SERVICE, "fixed") != "scan_pii_slow"))
    else:
        R["R7 Pyroscope (sin stack: se omite)"] = None
    chk("R8 worst_ticket: el ticket más caro y su función principal (scan_pii_slow)", lambda: (lambda t, f: f == "scan_pii_slow" and t.startswith("T-"))(*worst_ticket(PP)))
    chk("R9 cpu_guard: suspende la variante lenta y aprueba la corregida", lambda: (not cpu_guard(BASE.ms_per_ticket, BASE.ms_per_ticket)) and cpu_guard(run_fixed(N_P).ms_per_ticket, BASE.ms_per_ticket))
    ok = sum(v is True for v in R.values()); tot = sum(v is not None for v in R.values())
    for k, v in R.items():
        print(("✅" if v else ("⏭️ " if v is None else "❌")), k)
    print(f"\nPuntuación: {ok}/{tot} comprobaciones superadas" + ("" if live else " (R7 omitida: sin Pyroscope)"))
    return R

try:
    run_rubric()
except NameError as e:
    print("Define las funciones de los pasos primero:", e)

## Retos extra (nivel experto)

1. **Perfil diferencial de verdad**: sube `slow` y `fixed` con el SDK `pyroscope-io` y compáralos en Grafana (`Pyroscope → Compare`); explica por qué conviene normalizar.
2. **Mide la regex sola** (sin caché) en `run_fixed` y compárala con tu solución con caché: ¿cuánto vale optimizar la constante frente a no repetir trabajo? **Cambia `tokens` por un contador de tokens real** (p. ej. `tiktoken`, si lo instalas) y mide cómo cambia el reparto: ¿qué etapa pasa a dominar?
3. **Perfil *wall***: repite el diagnóstico con `mode="wall"` y `SUPPORTOPS_REAL_SLEEP_SCALE=0.02`; ¿qué aparece que el perfil de CPU no ve?
4. **Etiquetas por intención**: etiqueta las muestras con `supportops.intent` (cardinalidad 4) y responde: ¿qué tipo de ticket cuesta más CPU?
5. **CI**: escribe un test de `pytest` que falle si `ms_per_ticket` de `fast` supera 2× el valor de referencia medido en tu máquina.

## Reflexión

- ¿Qué habría pasado si optimizas primero lo que creías (serializar JSON)? Cuantifícalo con Amdahl (fracción medida de `serialize`).
- ¿Cuándo es **suficiente** esta CPU «gratis» frente a gastar el esfuerzo en reducir tokens (módulo 13)? Compara coste de cómputo de pods y coste del LLM por ticket.
- ¿Qué etiquetas pondrías en producción para detectar una regresión de CPU por **versión del agente** en el módulo 15?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable: redefine las funciones y vuelve a lanzar la rúbrica.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
import re
def top_function(stacks):
    fn, n = P.self_time(stacks).most_common(1)[0]
    return fn.split(".")[-1], n / sum(P.self_time(stacks).values())

def stage_report(stacks, stats):
    sp, st = P.stage_shares(stacks), stats.stage_share()
    return pd.DataFrame([{"etapa": s, "perfil": sp.get(s, 0.0), "cronómetro": st.get(s, 0.0)} for s in ("pii_scan", "tokens", "serialize", "embed_rerank")])

import functools
_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
_CARD = re.compile(r"\d(?:[ -]?\d){12,18}")

@functools.lru_cache(maxsize=65536)
def _scan_line(line):
    cards = sum(1 for m in _CARD.finditer(line) if P.luhn_ok(re.sub(r"\D", "", m.group())))
    return len(_EMAIL.findall(line)), cards

@functools.lru_cache(maxsize=65536)
def _tok_line(line):
    return sum(-(-len(w) // 4) for w in line.split())

def my_scan(text):                     # un correo o una tarjeta nunca cruzan un salto de línea: se puede cachear POR LÍNEA
    e = c = 0
    for line in text.split("\n"):
        a, b = _scan_line(line)
        e += a; c += b
    return {"emails": e, "cards": c}

def my_tokens(text):
    return sum(_tok_line(line) for line in text.split("\n"))

def run_fixed(n):
    return run_workload("slow", n=n, scan_fn=my_scan, tokens_fn=my_tokens)

def predict_speedup(fixed_share):
    return P.amdahl_speedup(fixed_share, 1e9)

def push_profile(prof, service, variant):
    P.PyroscopeClient().push_connect({"service_name": service, "variant": variant}, prof.to_pprof())

def read_top(service, variant):
    sel = '{service_name="%s",variant="%s"}' % (service, variant)
    c = P.PyroscopeClient(); c.wait_for(sel, timeout_s=90)
    return P.self_time(c.stacks(sel)).most_common(1)[0][0].split(".")[-1]

def worst_ticket(prof):
    t, n = prof.tag_values("ticket").most_common(1)[0]
    top = P.self_time(prof.stacks(ticket=t)).most_common(1)[0][0].split(".")[-1]
    return t, top

def cpu_guard(ms_per_ticket, baseline_ms, max_ratio=0.25):
    return ms_per_ticket <= max_ratio * baseline_ms

PF = SamplingProfiler(hz=300, mode="cpu"); run_workload("slow", n=N_P, profiler=PF, scan_fn=my_scan, tokens_fn=my_tokens)
if PYRO_UP:
    push_profile(PP, SERVICE, "slow"); push_profile(PF, SERVICE, "fixed")
print(top_function(ST_SLOW), worst_ticket(PP))
display(stage_report(ST_SLOW, BASE).round(3))
R = run_rubric()